# Semana 3 — Arquitectura por capas (bronze → silver → gold) con DuckDB

**Objetivo:** transformar bronze en una capa **silver** tipada, deduplicada y sin identificadores directos, y construir una capa **gold** con tablas listas para análisis: resumen por paciente, últimos signos vitales, actividad mensual y reingresos a 30 días.

**Producto real usado:** DuckDB, un motor SQL analítico embebido (corre dentro de Python, sin servidor).

**Al terminar tendrás:** `data/silver/*.parquet` y `data/gold/*.parquet`.

**Requisito:** semanas 1 y 2.

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

### Paso 1 — Conectar DuckDB y exponer bronze como vistas

**Explicación línea por línea**

- `%pip install -q "duckdb>=1.1"`: instala o actualiza DuckDB.
- `import duckdb`: cliente de Python.
- `con = duckdb.connect()`: abre una base **en memoria**. **Por qué en memoria y no un archivo en Drive:** el disco de Drive montado en Colab es una red lenta y no está pensado para los bloqueos de una base de datos. El patrón *lakehouse* separa almacenamiento (archivos Parquet en Drive) de cómputo (DuckDB efímero): cada semana recrea las tablas desde Parquet en segundos.
- `con.execute("SET TimeZone = 'UTC'")`: fija la zona horaria de la sesión. Sin esto, DuckDB muestra las fechas en la zona del equipo y dos personas verían horas distintas.
- `CREATE SCHEMA IF NOT EXISTS bronze`: un *schema* es un espacio de nombres (`bronze.patients` no choca con `silver.patients`).
- `CREATE OR REPLACE VIEW bronze.{t} AS SELECT * FROM read_parquet('...')`: una **vista** no copia datos; es una consulta guardada que lee el Parquet cada vez. `OR REPLACE` la hace idempotente.
- `.as_posix()`: convierte la ruta a formato con `/`, el que SQL espera en cualquier sistema operativo.
- `con.sql("SELECT ...").df()`: ejecuta SQL y devuelve un DataFrame de pandas. Para obtener un solo valor usaremos `con.execute(sql).fetchall()[0][0]` (primera fila, primera columna).

In [ ]:
%pip install -q "duckdb>=1.1"

In [ ]:
import duckdb

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
for d in (SILVER_DIR, GOLD_DIR):
    d.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")
for t in ["patients", "encounters", "conditions", "observations"]:
    ruta = (BRONZE_DIR / f"{t}.parquet").as_posix()
    con.execute(f"CREATE OR REPLACE VIEW bronze.{t} AS SELECT * FROM read_parquet('{ruta}')")

con.sql("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
""").df()

### Paso 2 — silver.patients: tipos explícitos y sin identificadores directos

**Explicación línea por línea (SQL)**

- `CREATE SCHEMA IF NOT EXISTS silver;` crea el espacio de la capa limpia.
- `CREATE OR REPLACE TABLE silver.patients AS SELECT ...`: crea una tabla física (en memoria) a partir de una consulta.
- `Id AS patient_id`: renombra a un estilo consistente (`snake_case`, sustantivo + `_id`). **En la práctica real** un estándar de nombres hace que cien tablas sean navegables.
- `TRY_CAST(BIRTHDATE AS DATE)`: convierte texto a fecha; si no puede, devuelve `NULL` en vez de abortar. `CAST` normal abortaría la consulta entera por una sola fila mala.
- `TRY_CAST(NULLIF(DEATHDATE, '') AS DATE)`: `NULLIF(x, '')` devuelve `NULL` si el texto está vacío; deja explícito que "vacío" significa "sin dato".
- `GENDER AS sex`: en Synthea `M`/`F` representa sexo registrado, no identidad de género; el nombre de columna lo deja claro.
- Columnas **no** seleccionadas: `SSN`, `DRIVERS`, `PASSPORT`, nombres, `ADDRESS`, `LAT`, `LON`, `ZIP`. Al no seleccionarlas, silver nace sin identificadores directos (inventario de la semana 1). El `patient_id` es un UUID sin significado: un **seudónimo**.
- `_ingested_at`: se conserva el linaje.
- `QUALIFY ROW_NUMBER() OVER (PARTITION BY Id ORDER BY _ingested_at DESC) = 1`: `ROW_NUMBER` numera las filas de cada `Id` empezando por la carga más reciente; `QUALIFY` filtra sobre el resultado de una función de ventana. Resultado: una fila por paciente aunque llegaran duplicados. Es el patrón estándar de deduplicación en SQL analítico.

In [ ]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS silver;
CREATE OR REPLACE TABLE silver.patients AS
SELECT
    Id                                  AS patient_id,
    TRY_CAST(BIRTHDATE AS DATE)         AS birth_date,
    TRY_CAST(NULLIF(DEATHDATE, '') AS DATE) AS death_date,
    GENDER                              AS sex,
    NULLIF(RACE, '')                    AS race,
    NULLIF(ETHNICITY, '')               AS ethnicity,
    NULLIF(MARITAL, '')                 AS marital_status,
    NULLIF(STATE, '')                   AS state,
    NULLIF(COUNTY, '')                  AS county,
    _ingested_at
FROM bronze.patients
QUALIFY ROW_NUMBER() OVER (PARTITION BY Id ORDER BY _ingested_at DESC) = 1
""")
con.sql("SELECT * FROM silver.patients LIMIT 5").df()

### Paso 3 — silver.encounters y silver.conditions

**Explicación línea por línea (SQL)**

- `TRY_CAST(START AS TIMESTAMPTZ) AT TIME ZONE 'UTC'`: primero interpreta el texto como instante con zona (`Z` = UTC) y luego lo expresa como hora UTC sin zona. **Por qué así:** si mañana llega un dato con `+10:00` (Brisbane), se convierte correctamente a UTC en vez de perder la diferencia.
- `NULLIF(REASONCODE, '')`: motivo de consulta; vacío en 74 % (lo viste en la semana 2), lo dejamos como `NULL`.
- `QUALIFY ROW_NUMBER() OVER (PARTITION BY Id ORDER BY START) = 1`: una fila por encuentro.
- En `silver.conditions`: `CODE AS snomed_code` documenta en el nombre que el código es SNOMED CT (en la versión 2021 de Synthea la columna `SYSTEM` no existe; lo sabemos por la documentación y lo verificaremos contra el servidor terminológico en la semana 5).
- `TRY_CAST(NULLIF(STOP, '') AS DATE) AS stop_date`: una condición sin fecha de fin está **activa**; ese significado de `NULL` se usará en gold.
- `SELECT DISTINCT`: las condiciones no traen identificador propio; `DISTINCT` elimina filas idénticas.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.encounters AS
SELECT
    Id                    AS encounter_id,
    PATIENT               AS patient_id,
    ENCOUNTERCLASS        AS encounter_class,
    CODE                  AS encounter_snomed_code,
    DESCRIPTION           AS encounter_description,
    TRY_CAST(START AS TIMESTAMPTZ) AT TIME ZONE 'UTC' AS start_ts,
    TRY_CAST(STOP AS TIMESTAMPTZ) AT TIME ZONE 'UTC'  AS stop_ts,
    NULLIF(REASONCODE, '') AS reason_snomed_code,
    ORGANIZATION          AS organization_id,
    PROVIDER              AS provider_id
FROM bronze.encounters
QUALIFY ROW_NUMBER() OVER (PARTITION BY Id ORDER BY START) = 1;

CREATE OR REPLACE TABLE silver.conditions AS
SELECT DISTINCT
    PATIENT                              AS patient_id,
    ENCOUNTER                            AS encounter_id,
    CODE                                 AS snomed_code,
    DESCRIPTION                          AS description,
    TRY_CAST(START AS DATE)              AS start_date,
    TRY_CAST(NULLIF(STOP, '') AS DATE)   AS stop_date
FROM bronze.conditions;
""")
con.sql("SELECT encounter_class, COUNT(*) AS n FROM silver.encounters GROUP BY 1 ORDER BY 2 DESC").df()

### Paso 4 — silver.observations: valores numéricos y duplicados exactos

**Explicación línea por línea**

- Primera consulta (`con.execute(...).fetchall()[0]` devuelve la primera fila como tupla): cuenta filas totales y filas distintas sobre las columnas de negocio. La diferencia son **duplicados exactos** (misma fecha, paciente, encuentro, código, descripción y valor). En esta muestra aparecen paneles metabólicos repetidos.
- `CREATE OR REPLACE TABLE silver.observations AS SELECT DISTINCT ...`: elimina duplicados exactos.
- `NULLIF(CATEGORY, '') AS category`: `vital-signs`, `laboratory`, `survey`...
- `CODE AS loinc_code`: Synthea codifica observaciones con LOINC; algunas (`QALY`, `DALY`, `QOLS`) son códigos propios del generador, no LOINC. Lo marcaremos en la semana 5.
- `CASE WHEN TYPE = 'numeric' THEN TRY_CAST(VALUE AS DOUBLE) END AS value_num`: solo convierte a número cuando el proveedor declara que es numérico; `DOUBLE` es número de coma flotante de 64 bits.
- `CASE WHEN TYPE = 'text' THEN VALUE END AS value_text`: separa valores numéricos y textuales en columnas distintas. **En la práctica real** es el mismo diseño de la tabla MEASUREMENT de OMOP (`value_as_number` y `value_source_value`), que usarás en la semana 6.
- `NULLIF(UNITS, '') AS unit`: unidades UCUM como `mm[Hg]` o `kg/m2`.

In [ ]:
dup = con.execute("""
    SELECT COUNT(*) AS filas,
           COUNT(DISTINCT (DATE, PATIENT, ENCOUNTER, CODE, DESCRIPTION, VALUE)) AS distintas
    FROM bronze.observations
""").fetchall()[0]
print(f"Filas: {dup[0]:,} | distintas: {dup[1]:,} | duplicados exactos: {dup[0] - dup[1]:,}")

con.execute("""
CREATE OR REPLACE TABLE silver.observations AS
SELECT DISTINCT
    TRY_CAST(DATE AS TIMESTAMPTZ) AT TIME ZONE 'UTC' AS obs_ts,
    PATIENT                    AS patient_id,
    NULLIF(ENCOUNTER, '')      AS encounter_id,
    NULLIF(CATEGORY, '')       AS category,
    CODE                       AS loinc_code,
    DESCRIPTION                AS description,
    CASE WHEN TYPE = 'numeric' THEN TRY_CAST(VALUE AS DOUBLE) END AS value_num,
    CASE WHEN TYPE = 'text' THEN VALUE END                       AS value_text,
    NULLIF(UNITS, '')          AS unit,
    TYPE                       AS value_type
FROM bronze.observations
""")
con.sql("SELECT COUNT(*) AS filas_silver FROM silver.observations").df()

### Paso 5 — Pruebas de datos sobre silver (antes de publicar)

En la industria, una capa no se publica si no pasa sus pruebas. Herramientas como dbt o las *expectations* de Databricks hacen esto; aquí lo hacemos con SQL y `assert`.

**Explicación línea por línea**

- `tests = {...}`: diccionario `nombre de la prueba → consulta SQL que debe devolver 0`. Escribir la prueba como "cuenta de filas que violan la regla" es la convención de dbt: 0 significa que pasa.
- `nulos_patient_id`: ningún paciente sin identificador.
- `fechas_no_convertidas`: filas donde el texto original no estaba vacío pero `TRY_CAST` devolvió `NULL`.
- `encuentros_huerfanos`: encuentros cuyo paciente no existe en silver (`LEFT JOIN ... WHERE p.patient_id IS NULL` es el patrón clásico para encontrar huérfanos).
- `conciliacion_patients`: diferencia de conteo bronze vs silver (`ABS` para valor absoluto).
- `for nombre, sql in tests.items(): n = con.execute(sql).fetchall()[0][0]`: ejecuta cada prueba; `fetchall()` trae todas las filas del resultado (aquí una), `[0]` toma la primera fila y el segundo `[0]` su primer valor. Se consume el resultado completo porque DuckDB 2.x no permite lanzar otra consulta mientras queda un resultado a medio leer; así el código funciona igual en DuckDB 1.x y 2.x.
- `assert not fallidas`: si alguna prueba falla, el notebook se detiene y **no** se exporta nada.

In [ ]:
tests = {
    "nulos_patient_id": "SELECT COUNT(*) FROM silver.patients WHERE patient_id IS NULL",
    "fechas_no_convertidas": """
        SELECT COUNT(*) FROM bronze.encounters
        WHERE START <> '' AND TRY_CAST(START AS TIMESTAMPTZ) IS NULL""",
    "encuentros_huerfanos": """
        SELECT COUNT(*) FROM silver.encounters e
        LEFT JOIN silver.patients p USING (patient_id)
        WHERE p.patient_id IS NULL""",
    "conciliacion_patients": """
        SELECT ABS((SELECT COUNT(*) FROM bronze.patients) - (SELECT COUNT(*) FROM silver.patients))""",
    "conciliacion_encounters": """
        SELECT ABS((SELECT COUNT(*) FROM bronze.encounters) - (SELECT COUNT(*) FROM silver.encounters))""",
}
fallidas = []
for nombre, sql in tests.items():
    n = con.execute(sql).fetchall()[0][0]
    print(f"{nombre:<26} {'PASS' if n == 0 else 'FAIL'} ({n})")
    if n:
        fallidas.append(nombre)
assert not fallidas, f"Pruebas fallidas: {fallidas}"

### Paso 6 — gold.patient_summary: una fila por paciente

**Explicación línea por línea (SQL)**

- `WITH ref AS (...)`: una *CTE* (expresión de tabla común) es una subconsulta con nombre. `ref_date` es la fecha del último encuentro del dataset: la "fecha de corte" de los datos. **Por qué no `CURRENT_DATE`:** los datos terminan en noviembre de 2021; calcular la edad a hoy inflaría todas las edades y el resultado cambiaría cada día (no reproducible).
- `enc AS (...)`: agrega encuentros por paciente. `COUNT(*) FILTER (WHERE ...)` cuenta solo las filas que cumplen la condición; es más legible que `SUM(CASE WHEN ...)`.
- `cond AS (...)`: número de condiciones distintas activas (`stop_date IS NULL`).
- `FLOOR(date_diff('day', p.birth_date, COALESCE(p.death_date, ref.ref_date)) / 365.25)`: edad en años cumplidos a la muerte o a la fecha de corte. `COALESCE` toma el primer valor no nulo. Se usa la diferencia en días porque `date_diff('year', ...)` cuenta cambios de año calendario, no años cumplidos.
- `CROSS JOIN ref`: `ref` tiene una sola fila; el cruce la pega a cada paciente.
- `LEFT JOIN enc ... LEFT JOIN cond`: conserva pacientes aunque no tengan condiciones; `COALESCE(..., 0)` convierte ese `NULL` en 0.

In [ ]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS gold;
CREATE OR REPLACE TABLE gold.patient_summary AS
WITH ref AS (
    SELECT MAX(stop_ts)::DATE AS ref_date FROM silver.encounters
),
enc AS (
    SELECT patient_id,
           COUNT(*)                                                         AS n_encounters,
           COUNT(*) FILTER (WHERE encounter_class IN ('emergency', 'urgentcare')) AS n_acute,
           COUNT(*) FILTER (WHERE encounter_class = 'inpatient')            AS n_inpatient,
           MIN(start_ts)::DATE                                              AS first_encounter,
           MAX(start_ts)::DATE                                              AS last_encounter
    FROM silver.encounters
    GROUP BY patient_id
),
cond AS (
    SELECT patient_id, COUNT(DISTINCT snomed_code) AS n_active_conditions
    FROM silver.conditions
    WHERE stop_date IS NULL
    GROUP BY patient_id
)
SELECT p.patient_id, p.sex, p.birth_date, p.death_date,
       FLOOR(date_diff('day', p.birth_date, COALESCE(p.death_date, ref.ref_date)) / 365.25)::INTEGER AS age_years,
       p.death_date IS NULL AS alive_at_ref,
       enc.n_encounters, enc.n_acute, enc.n_inpatient, enc.first_encounter, enc.last_encounter,
       COALESCE(cond.n_active_conditions, 0) AS n_active_conditions,
       ref.ref_date
FROM silver.patients p
CROSS JOIN ref
LEFT JOIN enc USING (patient_id)
LEFT JOIN cond USING (patient_id)
""")
con.sql("SELECT * FROM gold.patient_summary ORDER BY n_encounters DESC LIMIT 5").df()

### Paso 7 — gold.vitals_latest: último valor por paciente (funciones de ventana)

**Explicación línea por línea (SQL)**

- `WITH ultimos AS (...)`: para cada paciente y código toma la medición más reciente.
- `WHERE loinc_code IN ('8480-6', '8462-4', '39156-5') AND value_num IS NOT NULL`: sistólica, diastólica e IMC con valor.
- `QUALIFY ROW_NUMBER() OVER (PARTITION BY patient_id, loinc_code ORDER BY obs_ts DESC) = 1`: número 1 = la más reciente dentro de cada (paciente, código).
- `MAX(value_num) FILTER (WHERE loinc_code = '8480-6') AS sbp_last`: *pivoteo* con agregación condicional; convierte filas (una por código) en columnas. Como hay una sola fila por código, `MAX` solo "extrae" ese valor.
- `GROUP BY patient_id`: una fila por paciente. **En la práctica real** esta tabla es una *feature table*: insumo directo para un modelo o un tablero clínico.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE gold.vitals_latest AS
WITH ultimos AS (
    SELECT patient_id, loinc_code, value_num, obs_ts
    FROM silver.observations
    WHERE loinc_code IN ('8480-6', '8462-4', '39156-5') AND value_num IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY patient_id, loinc_code ORDER BY obs_ts DESC) = 1
)
SELECT patient_id,
       MAX(value_num) FILTER (WHERE loinc_code = '8480-6')  AS sbp_last,
       MAX(value_num) FILTER (WHERE loinc_code = '8462-4')  AS dbp_last,
       MAX(value_num) FILTER (WHERE loinc_code = '39156-5') AS bmi_last,
       MAX(obs_ts)                                         AS last_vitals_ts
FROM ultimos
GROUP BY patient_id
""")
con.sql("SELECT * FROM gold.vitals_latest LIMIT 5").df()

### Paso 8 — gold.encounters_monthly y un indicador clásico: reingreso a 30 días

**Explicación línea por línea (SQL)**

- `date_trunc('month', start_ts)::DATE AS month`: lleva cada fecha al primer día de su mes; es la forma estándar de agrupar series de tiempo.
- `GROUP BY 1, 2`: agrupa por la primera y segunda columna del `SELECT`.
- Consulta de reingresos: `WITH ip AS (...)` toma solo hospitalizaciones (`inpatient`).
- `LEAD(start_ts) OVER (PARTITION BY patient_id ORDER BY start_ts) AS next_start`: `LEAD` trae el valor de la **siguiente** fila del mismo paciente: la fecha de su próxima hospitalización.
- `next_start <= stop_ts + INTERVAL 30 DAY`: hay reingreso si la siguiente hospitalización empieza dentro de los 30 días posteriores al alta. **Por qué importa:** la tasa de reingreso a 30 días es un indicador de calidad hospitalaria ampliamente usado por sistemas de salud y aseguradoras. Esta es una definición simplificada (no excluye reingresos programados).
- `ROUND(100.0 * ... / COUNT(*), 1)`: porcentaje; `100.0` (con decimal) evita la división entera.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE gold.encounters_monthly AS
SELECT date_trunc('month', start_ts)::DATE AS month,
       encounter_class,
       COUNT(*)                   AS n_encounters,
       COUNT(DISTINCT patient_id) AS n_patients
FROM silver.encounters
GROUP BY 1, 2
""")

con.sql("""
WITH ip AS (
    SELECT patient_id, start_ts, stop_ts,
           LEAD(start_ts) OVER (PARTITION BY patient_id ORDER BY start_ts) AS next_start
    FROM silver.encounters
    WHERE encounter_class = 'inpatient'
)
SELECT COUNT(*) AS hospitalizaciones,
       COUNT(*) FILTER (WHERE next_start <= stop_ts + INTERVAL 30 DAY) AS reingresos_30d,
       ROUND(100.0 * COUNT(*) FILTER (WHERE next_start <= stop_ts + INTERVAL 30 DAY) / COUNT(*), 1) AS pct
FROM ip
""").df()

### Paso 9 — Una pregunta clínica sobre gold: prevalencia de hipertensión por sexo y edad

**Explicación línea por línea (SQL)**

- `htn AS (SELECT DISTINCT patient_id FROM silver.conditions WHERE snomed_code = '59621000')`: pacientes con el código SNOMED CT 59621000 (hipertensión esencial) alguna vez registrado. En la semana 5 reemplazarás este código único por un **conjunto de conceptos** obtenido del servidor terminológico.
- `CASE WHEN age_years < 40 THEN '18-39' ...`: grupos de edad.
- `WHERE s.alive_at_ref AND s.age_years >= 18`: adultos vivos a la fecha de corte; el denominador importa tanto como el numerador.
- `AVG(CASE WHEN h.patient_id IS NOT NULL THEN 1.0 ELSE 0 END)`: proporción de pacientes con hipertensión. Es una **prevalencia de período registrada**: mide diagnósticos registrados, no enfermedad real (recuerda a Agniel et al.).

In [ ]:
con.sql("""
WITH htn AS (
    SELECT DISTINCT patient_id FROM silver.conditions WHERE snomed_code = '59621000'
)
SELECT s.sex,
       CASE WHEN s.age_years < 40 THEN '18-39'
            WHEN s.age_years < 65 THEN '40-64'
            ELSE '65+' END AS age_group,
       COUNT(*) AS n,
       ROUND(100 * AVG(CASE WHEN h.patient_id IS NOT NULL THEN 1.0 ELSE 0 END), 1) AS pct_htn
FROM gold.patient_summary s
LEFT JOIN htn h USING (patient_id)
WHERE s.alive_at_ref AND s.age_years >= 18
GROUP BY 1, 2
ORDER BY 1, 2
""").df()

### Paso 10 — Publicar silver y gold como Parquet

**Explicación línea por línea**

- `def export(schema, table, folder):` escribe una tabla de DuckDB a Parquet.
- `COPY {schema}.{table} TO '...' (FORMAT parquet, COMPRESSION zstd)`: `COPY` es la forma nativa y más rápida de exportar; `zstd` comprime bien y es estándar en la industria.
- El bucle exporta 4 tablas silver y 3 gold.
- `con.close()`: libera la memoria. La próxima semana recrearás todo desde estos Parquet.

In [ ]:
def export(schema, table, folder):
    destino = (folder / f"{table}.parquet").as_posix()
    con.execute(f"COPY {schema}.{table} TO '{destino}' (FORMAT parquet, COMPRESSION zstd)")
    n = con.execute(f"SELECT COUNT(*) FROM {schema}.{table}").fetchall()[0][0]
    print(f"{schema}.{table:<20} {n:>9,} filas → {destino}")


for t in ["patients", "encounters", "conditions", "observations"]:
    export("silver", t, SILVER_DIR)
for t in ["patient_summary", "vitals_latest", "encounters_monthly"]:
    export("gold", t, GOLD_DIR)
con.close()

### Autoevaluación

1. ¿Por qué silver se construye desde bronze y no directamente desde el CSV?
2. Explica con tus palabras qué hace `QUALIFY ROW_NUMBER() OVER (...) = 1`.
3. ¿Por qué la edad se calcula a la fecha de corte de los datos y no a hoy?
4. La tasa de reingreso a 30 días de este notebook es "simplificada". Nombra dos exclusiones que una definición oficial suele aplicar.

**Commit sugerido:** `feat(s03): capas silver y gold en DuckDB con pruebas de datos`